# Road Network Comparison
This notebook refactors the visualization to use pre-processed road network segments and compares them with the raw KMZ road backbone.

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from folium import LayerControl
import fiona
import os

# Enable KML driver for KMZ loading
fiona.drvsupport.supported_drivers['KML'] = 'rw'

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
# Paths
processed_path = '../data/processed/integrated_road_network.parquet'
kmz_path = '../data/processed/backbone_roads.parquet'

# 1. Load Processed Road Network
print("Loading processed road network...")
gdf_processed = gpd.read_parquet(processed_path)
if gdf_processed.crs != 4326:
    gdf_processed = gdf_processed.to_crs(epsg=4326)
print(f"Loaded {len(gdf_processed)} processed segments.")

# 2. Load KMZ Roads
print("Loading KMZ road backbone...")
# We use zip:// protocol to read the KMZ file
gdf_kmz = gpd.read_parquet(kmz_path)

if gdf_kmz.crs != 4326:
    gdf_kmz = gdf_kmz.to_crs(epsg=4326)
print(f"Loaded {len(gdf_kmz)} KMZ road lines.")

gdf_processed.head()

## 2. Visualizations

### 2.1 Raw Roads (Red)

In [ ]:
center = [40.4168, -3.7038]

In [ ]:
m1 = folium.Map(location=center, zoom_start=6, tiles="cartodbpositron")

# Add KMZ Roads
folium.GeoJson(
    gdf_kmz,
    name="KMZ Backbone",
    style_function=lambda x: {'color': 'red', 'weight': 2, 'opacity': 0.7}    
).add_to(m1)

m1

### 2.2 Processed Road Network (Blue)

In [ ]:

# Calculate center for maps
center = [gdf_processed.geometry.centroid.y.mean(), gdf_processed.geometry.centroid.x.mean()]

m2 = folium.Map(location=center, zoom_start=6, tiles="cartodbpositron")

# Add Processed Segments
folium.GeoJson(
    gdf_processed,
    name="Processed Network",
    style_function=lambda x: {'color': 'blue', 'weight': 2, 'opacity': 0.7},
    tooltip=folium.GeoJsonTooltip(fields=["backbone_id", "original_segment_count"], aliases=["Backbone ID:", "Original Segments:"])
).add_to(m2)

m2


### 2.3 Comparative Visualization (Overlap)

In [ ]:
m3 = folium.Map(location=center, zoom_start=6, tiles="cartodbpositron")

# Create groups for LayerControl
fg_processed = folium.FeatureGroup(name="Processed Network (Blue)").add_to(m3)
fg_kmz = folium.FeatureGroup(name="KMZ Backbone (Red)").add_to(m3)

# Add Processed to group
folium.GeoJson(
    gdf_processed,
    style_function=lambda x: {'color': 'blue', 'weight': 3, 'opacity': 0.6}
).add_to(fg_processed)

# Add KMZ to group
folium.GeoJson(
    gdf_kmz,
    style_function=lambda x: {'color': 'red', 'weight': 2, 'opacity': 0.8}
).add_to(fg_kmz)

folium.LayerControl().add_to(m3)
m3

In [ ]:
import branca.colormap as cm

# Create colormap based on traffic intensity
max_traffic = gdf_processed['total_20241019'].max()
min_traffic = gdf_processed['total_20241019'].min()
colormap = cm.linear.YlOrRd_09.scale(min_traffic, max_traffic)
colormap.caption = 'Traffic Intensity (2024-10-19)'

# Create map
m4 = folium.Map(location=[40.4168, -3.7038], zoom_start=6, tiles='cartodbpositron')

# Add colormap to map
colormap.add_to(m4)

def style_fn(feature):
    traffic = feature['properties'].get('total_20241019', 0)
    return {
        'fillColor': colormap(traffic),
        'color': colormap(traffic),
        'weight': 4,
        'fillOpacity': 0.8
    }

folium.GeoJson(
    gdf_processed,
    style_function=style_fn,
    tooltip=folium.GeoJsonTooltip(fields=['backbone_id', 'total_20241019'], aliases=['Backbone ID:', 'Traffic:'])
).add_to(m4)

m4